# 🧪 Lab 03 — Fixed SRIDs, `0`, and the Spatial Multiverse 🛰️🌌

Spark now understands spatial types. Great.

Now we ask a nastier question:

> **How strict is the SRID contract, what exactly does `0` mean, why does `GEOMETRY(ANY)` exist, and what happens when the multiverse hits disk?**

This lab is intentionally narrower than Lab 02. We are no longer comparing Geometry with Geography. We are dissecting **Geometry's SRID contract itself**.

### 🎯 Mission objectives

We will prove that:

- `GEOMETRY(4326)` and `GEOMETRY(3857)` are fixed-SRID contracts;
- `GEOMETRY(0)` is a **fixed SRID 0** type, not a synonym for `ANY`;
- `ST_GeomFromWKB(wkb)` defaults to SRID `0`;
- Spark rejects a value whose SRID contradicts a fixed-SRID column;
- combining fixed-SRID Geometry values can widen to `GEOMETRY(ANY)`;
- the individual rows inside `GEOMETRY(ANY)` keep their original SRIDs;
- Spark does **not** reproject anything when it widens to `ANY`;
- fixed-SRID Geometry survives a Parquet round trip;
- `GEOMETRY(ANY)` is rejected at the Parquet storage boundary because persistent spatial columns require one fixed SRID.

No Sedona. No Delta extension. No Iceberg runtime.

The Parquet experiment is real. The Delta/Iceberg rule mentioned in the article remains documentation-backed because those storage engines are not part of stock local Spark.

## 0 — Pre-flight checks 🛰️

Target environment:

```text
PySpark  4.2.0
Spark    4.2.0
Java     17+
```

As before, the notebook includes the Windows/local bootstrap needed when `PYSPARK_SUBMIT_ARGS` is missing.

In [1]:
import sys, json, warnings, struct,tempfile, shutil
from pathlib import Path


warnings.filterwarnings(
    "ignore",
    message=r"PySpark does not yet fully support pandas >= 3\.0\.0.*",
    category=FutureWarning,
)

import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, GeometryType

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab-03-fixed-srids-zero-and-the-spatial-multiverse")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

java_version = spark.sparkContext._jvm.java.lang.System.getProperty("java.version")
fingerprint = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "java": java_version,
    "geospatial_enabled": spark.conf.get("spark.sql.geospatial.enabled"),
}

print("🚀 Runtime fingerprint")
print(json.dumps(fingerprint, indent=2))

assert pyspark.__version__ == "4.2.0"
assert spark.version == "4.2.0"
assert int(java_version.split(".")[0]) >= 17
assert fingerprint["geospatial_enabled"].lower() == "true"

print("\n✅ Multiverse containment field online.")

🚀 Runtime fingerprint
{
  "python": "3.14.0",
  "pyspark": "4.2.0",
  "spark": "4.2.0",
  "java": "17.0.19",
  "geospatial_enabled": "true"
}

✅ Multiverse containment field online.


# 1 — Four Types, Four Contracts 🧭

The section starts with:

```text
GEOMETRY(4326)
GEOMETRY(3857)
GEOMETRY(0)
GEOMETRY(ANY)
```

They are not four spellings of the same type.

They describe four different promises:

```text
GEOMETRY(4326)
→ every value follows SRID 4326

GEOMETRY(3857)
→ every value follows SRID 3857

GEOMETRY(0)
→ every value follows fixed SRID 0
  (unspecified Cartesian reference)

GEOMETRY(ANY)
→ rows may carry different valid SRIDs
```

First, let's ask PySpark's type system directly.

In [2]:
types = {
    "4326": GeometryType(4326),
    "3857": GeometryType(3857),
    "0": GeometryType(0),
    "ANY": GeometryType("ANY"),
}

print("🧭 Geometry type contracts")
for label, dtype in types.items():
    print(f"  ├─ GEOMETRY({label:<4}) → {dtype.simpleString()}")

assert types["4326"].simpleString().lower() == "geometry(4326)"
assert types["3857"].simpleString().lower() == "geometry(3857)"
assert types["0"].simpleString().lower() == "geometry(0)"
assert types["ANY"].simpleString().lower() == "geometry(any)"

lab_results = {
    "types": {k: v.simpleString() for k, v in types.items()},
}

🧭 Geometry type contracts
  ├─ GEOMETRY(4326) → geometry(4326)
  ├─ GEOMETRY(3857) → geometry(3857)
  ├─ GEOMETRY(0   ) → geometry(0)
  ├─ GEOMETRY(ANY ) → geometry(any)


# 2 — SRID `0`: Unknown Reference, Not Unlimited Freedom 🌑

This is an easy one to misunderstand.

Spark's one-argument constructor:

```python
st_geomfromwkb(wkb)
```

returns:

```text
GEOMETRY(0)
```

That means:

> **The shape is Cartesian Geometry, but no known coordinate reference has been attached.**

It does **not** mean:

> “Every row may use a different CRS.”

That second idea is `GEOMETRY(ANY)`.

Let's prove that `0` is a real fixed SRID carried by the value.

In [3]:
def wkb_point(x, y):
    return struct.pack("<BIdd", 1, 1, float(x), float(y))

local_point = spark.createDataFrame(
    [(1, wkb_point(125.0, 88.0))],
    ["id", "wkb"],
).select(
    "id",
    F.st_geomfromwkb("wkb").alias("geom"),
)

zero_row = local_point.select(
    F.expr("typeof(geom)").alias("spark_type"),
    F.st_srid("geom").alias("srid"),
).first()

print("🌑 Default Geometry constructor")
print(f"  ├─ type : {zero_row.spark_type}")
print(f"  └─ SRID : {zero_row.srid}")

assert zero_row.spark_type.lower() == "geometry(0)"
assert zero_row.srid == 0
assert zero_row.spark_type.lower() != "geometry(any)"

lab_results.update({
    "default_geometry_type": zero_row.spark_type,
    "default_geometry_srid": zero_row.srid,
    "zero_is_any": False,
})

🌑 Default Geometry constructor
  ├─ type : geometry(0)
  └─ SRID : 0


### Why would SRID `0` ever be legitimate?

Plenty of spatial systems are Cartesian without belonging to a registered Earth CRS:

```text
factory floor coordinates
image coordinates
robot simulation space
engineering grids
game worlds
```

For those, `GEOMETRY(0)` can be honest.

But if your coordinates really came from WGS 84 or Web Mercator and the CRS disappeared upstream, `0` is not a feature.

It's evidence.

# 3 — Fixed SRID Means Fixed 🥊

Now we create a real native Geometry carrying SRID `4326`.

Then we try to force that value into a schema declaring:

```text
GEOMETRY(3857)
```

If Spark accepts it, the fixed-SRID type contract would be meaningless.

Let's see whether the schema fights back.

In [4]:
g4326_value = (
    spark.createDataFrame(
        [(wkb_point(-3.7038, 40.4168),)],
        ["wkb"],
    )
    .select(F.st_geomfromwkb("wkb", 4326).alias("geom"))
    .first()["geom"]
)

fixed_3857_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("geom", GeometryType(3857), False),
])

fixed_mismatch_error = None

try:
    mismatch = spark.createDataFrame(
        [(1, g4326_value)],
        schema=fixed_3857_schema,
    )
    mismatch.collect()
except Exception as exc:
    fixed_mismatch_error = exc

def compact_error(exc):
    lines = [line.strip() for line in str(exc).splitlines() if line.strip()]
    return lines[0] if lines else repr(exc)

print("🥊 Fixed-SRID mismatch")
print("  ├─ value carries : SRID 4326")
print("  ├─ column demands: GEOMETRY(3857)")
print(f"  └─ accepted      : {fixed_mismatch_error is None}")

if fixed_mismatch_error:
    print(f"     error         : {compact_error(fixed_mismatch_error)}")

assert fixed_mismatch_error is not None

lab_results.update({
    "fixed_mismatch_rejected": True,
    "fixed_mismatch_error_type": type(fixed_mismatch_error).__name__,
})

🥊 Fixed-SRID mismatch
  ├─ value carries : SRID 4326
  ├─ column demands: GEOMETRY(3857)
  └─ accepted      : False
     error         : [GEO_ENCODER_SRID_MISMATCH_ERROR] Failed to encode GEOMETRY value because provided SRID 4326 of a value to encode does not match type SRID: 3857. SQLSTATE: 42K09


# 4 — Open the Multiverse: `GEOMETRY(ANY)` 🌌

Now the useful monstrosity.

We create three datasets:

```text
A → GEOMETRY(4326)
B → GEOMETRY(3857)
C → GEOMETRY(25830)
```

Then `UNION` them.

Spark needs one output datatype.

It cannot honestly claim every row is 4326, 3857, or 25830.

So what does Catalyst choose?

This is the reason `ANY` exists.

In [5]:
def one_geometry(row_id, x, y, srid):
    return (
        spark.createDataFrame(
            [(row_id, wkb_point(x, y))],
            ["id", "wkb"],
        )
        .select(
            "id",
            F.st_geomfromwkb("wkb", srid).alias("geom"),
        )
    )

g4326 = one_geometry(1, -3.7038, 40.4168, 4326)
g3857 = one_geometry(2, -412305.13, 4926696.67, 3857)
g25830 = one_geometry(3, 440000.0, 4474000.0, 25830)

mixed = (
    g4326
    .unionByName(g3857)
    .unionByName(g25830)
)

print("🌌 Mixed-SRID UNION")
mixed.printSchema()

mixed_telemetry = mixed.select(
    "id",
    F.expr("typeof(geom)").alias("column_type"),
    F.st_srid("geom").alias("value_srid"),
    F.hex(F.st_asbinary("geom")).alias("wkb_hex"),
).orderBy("id")

mixed_telemetry.show(truncate=False)

mixed_rows = mixed_telemetry.collect()
mixed_type = mixed_rows[0].column_type.lower()
mixed_srids = [r.value_srid for r in mixed_rows]

assert mixed_type == "geometry(any)"
assert mixed_srids == [4326, 3857, 25830]

print("📡 Catalyst verdict:")
print(f"  ├─ output type : {mixed_type}")
print(f"  └─ row SRIDs   : {mixed_srids}")

lab_results.update({
    "mixed_type": mixed_type,
    "mixed_srids": mixed_srids,
})

🌌 Mixed-SRID UNION
root
 |-- id: long (nullable = true)
 |-- geom: geometry(any) (nullable = true)

+---+-------------+----------+------------------------------------------+
|id |column_type  |value_srid|wkb_hex                                   |
+---+-------------+----------+------------------------------------------+
|1  |geometry(any)|4326      |0101000000FE65F7E461A10DC0857CD0B359354440|
|2  |geometry(any)|3857      |010100000052B81E85442A19C1AE47E12A3ACB5241|
|3  |geometry(any)|25830     |01010000000000000000DB1A410000000024115141|
+---+-------------+----------+------------------------------------------+

📡 Catalyst verdict:
  ├─ output type : geometry(any)
  └─ row SRIDs   : [4326, 3857, 25830]


### `ANY` did not reproject anything

This is crucial.

The output column became:

```text
GEOMETRY(ANY)
```

while the individual rows still carry:

```text
4326
3857
25830
```

Spark did not convert the coordinates.

It widened the **type contract** so the schema could honestly describe the heterogeneous result.

Notice what did **not** happen:

```text
4326 → stayed 4326
3857 → stayed 3857
25830 → stayed 25830
```

Spark widened the **column type** to `GEOMETRY(ANY)`. It did not alter the coordinates and it did not transform any row into a common CRS.

> **`ANY` is type honesty. Not spatial magic.**

# 5 — The Multiverse Meets Parquet 📦💥

Now for the experiment that justifies this lab.

`GEOMETRY(ANY)` works as an in-memory/query type.

But persistent spatial formats need a column-level spatial reference.

We will test both sides:

```text
GEOMETRY(4326)
→ write to Parquet
→ read it back
→ should remain fixed SRID

GEOMETRY(ANY)
→ write to Parquet
→ should be rejected
```

The expected contrast is brutally simple:

```text
GEOMETRY(4326) → Parquet ✅
GEOMETRY(ANY)  → Parquet ❌
```

This is the point where an elegant query-time abstraction runs into a storage format asking:

> **“Lovely. Which CRS does this column actually use?”**

In [6]:
tmp_root = Path(tempfile.mkdtemp(prefix="spark_spatial_multiverse_"))
fixed_path = tmp_root / "fixed_4326"
mixed_path = tmp_root / "mixed_any"

print(f"📦 Temporary evidence directory: {tmp_root}")

# ------------------------------------------------------------
# A) Fixed SRID → Parquet should work
# ------------------------------------------------------------
g4326.write.mode("overwrite").parquet(str(fixed_path))

fixed_read = spark.read.parquet(str(fixed_path))

fixed_read_row = fixed_read.select(
    F.expr("typeof(geom)").alias("column_type"),
    F.st_srid("geom").alias("value_srid"),
).first()

print("\n✅ Fixed-SRID Parquet round trip")
print(f"  ├─ read type : {fixed_read_row.column_type}")
print(f"  └─ row SRID  : {fixed_read_row.value_srid}")

assert fixed_read_row.column_type.lower() == "geometry(4326)"
assert fixed_read_row.value_srid == 4326

# ------------------------------------------------------------
# B) Mixed SRID → Parquet should fail
# ------------------------------------------------------------
any_write_error = None

spark.sparkContext.setLogLevel("OFF")
try:
    mixed.write.mode("overwrite").parquet(str(mixed_path))
except Exception as exc:
    any_write_error = exc
finally:
    spark.sparkContext.setLogLevel("ERROR")

print("\n🌌 GEOMETRY(ANY) Parquet write")
print(f"  ├─ accepted : {any_write_error is None}")
if any_write_error:
    print(f"  └─ error    : {compact_error(any_write_error)}")

assert any_write_error is not None, (
    "Expected Spark 4.2 to reject GEOMETRY(ANY) persistence to Parquet."
)

lab_results.update({
    "fixed_parquet_roundtrip": True,
    "fixed_parquet_type": fixed_read_row.column_type,
    "fixed_parquet_srid": fixed_read_row.value_srid,
    "any_parquet_rejected": True,
    "any_parquet_error_type": type(any_write_error).__name__,
    "any_parquet_error_message": compact_error(any_write_error),
})

# Keep the notebook polite.
shutil.rmtree(tmp_root, ignore_errors=True)

📦 Temporary evidence directory: C:\Users\ANGEL~1.ALV\AppData\Local\Temp\spark_spatial_multiverse_px0ay9xe

✅ Fixed-SRID Parquet round trip
  ├─ read type : geometry(4326)
  └─ row SRID  : 4326

🌌 GEOMETRY(ANY) Parquet write
  ├─ accepted : False
  └─ error    : [UNSUPPORTED_DATA_TYPE_FOR_DATASOURCE] The Parquet datasource doesn't support the column `geom` of the type "GEOMETRY(ANY)". SQLSTATE: 0A000


# 6 — What This Lab Proves, and What It Does Not 🧠

The Parquet result is direct experimental evidence.

This notebook proves:

```text
fixed-SRID Geometry can be persisted to Parquet
GEOMETRY(ANY) cannot
```

The article also mentions **Delta** and **Iceberg**.

Spark's 4.2 geospatial documentation states the same fixed-SRID storage rule for Parquet, Delta, and Iceberg, but this notebook does **not** load Delta or Iceberg runtimes just to manufacture redundant evidence.

So keep the epistemic boundary clean:

```text
Parquet  → ✅ runtime-proven in this notebook
Delta    → 📚 Spark-documented rule
Iceberg  → 📚 Spark-documented rule
```

The article can state the Delta/Iceberg rule because Spark documents it, but this notebook does **not** pretend to have executed those storage engines.

No random jars.

No dependency soup.

No trust-me-bro interoperability benchmark.

# 📊 Post-Lab Analysis — Let the Evidence Write the Verdict

The next cell generates the final analysis from the values captured during this execution.

If Spark behaves differently, an assertion above stops the notebook before this cell gets to invent a successful mission.

In [7]:
from IPython.display import Markdown, display

analysis = f"""
# 📊 Post-Lab Analysis: The Multiverse Is Fine Until Somebody Asks for a File

We tested four different Geometry contracts:

```text
{lab_results['types']['4326']}
{lab_results['types']['3857']}
{lab_results['types']['0']}
{lab_results['types']['ANY']}
```

They behaved as four different promises, not cosmetic aliases.

### 1. SRID `0` Is Fixed, Not Wildcard

Calling `ST_GeomFromWKB(wkb)` without an SRID produced:

**`{lab_results['default_geometry_type']}`**

with value SRID:

**`{lab_results['default_geometry_srid']}`**

That is a fixed “unspecified Cartesian reference” contract.

It is **not** `GEOMETRY(ANY)`.

### 2. Fixed SRID Actually Means Fixed

A Geometry value carrying SRID `4326` was forced into a `GEOMETRY(3857)` schema.

Spark rejected it:

**{lab_results['fixed_mismatch_rejected']}**

The failure type was:

**`{lab_results['fixed_mismatch_error_type']}`**

That is the point of embedding the SRID in the datatype: the schema can now reject a contradiction instead of silently admiring some bytes.

### 3. `ANY` Is Catalyst Admitting the Truth

Unioning Geometry values with SRIDs:

```text
{lab_results['mixed_srids']}
```

produced the common output type:

**`{lab_results['mixed_type']}`**

The rows kept their original SRIDs.

```text
4326  → 4326
3857  → 3857
25830 → 25830
```

No reprojection happened. Spark changed the **common column type**, not the spatial coordinates.

Spark simply widened the type so the schema could describe the mixed result honestly.

### 4. Then the Multiverse Hit Disk

The storage boundary reduced the whole experiment to one very useful table:

```text
GEOMETRY(4326) → Parquet ✅
GEOMETRY(ANY)  → Parquet ❌
```

The fixed-SRID dataset successfully completed a Parquet round trip as:

**`{lab_results['fixed_parquet_type']}`**

with SRID:

**`{lab_results['fixed_parquet_srid']}`**

But writing `GEOMETRY(ANY)` to Parquet was rejected:

**{lab_results['any_parquet_rejected']}**

with:

**`{lab_results['any_parquet_error_type']}`**

This is the architectural boundary the section describes.

Query-time type inference can say:

> “These are all geometries, but the rows disagree about the SRID.”

Persistent spatial storage eventually has to say:

> “This column has **this** spatial reference.”

> ## 🚀 Mission Verdict
> **Fixed SRIDs are contracts, `0` is an explicit unknown Cartesian reference, and `ANY` is a query-time escape hatch for honest mixed-SRID results.**
>
> `ANY` does not transform anything. It simply preserves disagreement.
>
> And when that disagreement reaches Parquet, storage refuses to pretend the column has one CRS.
>
> **The multiverse is perfectly legal in memory. Disk makes you pick a universe.**

### Evidence boundary

This notebook **runtime-proves Parquet** behavior. The same fixed-SRID persistence rule for **Delta** and **Iceberg** is stated in Spark 4.2 documentation, but those engines are not executed here.
"""

display(Markdown(analysis))


# 📊 Post-Lab Analysis: The Multiverse Is Fine Until Somebody Asks for a File

We tested four different Geometry contracts:

```text
geometry(4326)
geometry(3857)
geometry(0)
geometry(any)
```

They behaved as four different promises, not cosmetic aliases.

### 1. SRID `0` Is Fixed, Not Wildcard

Calling `ST_GeomFromWKB(wkb)` without an SRID produced:

**`geometry(0)`**

with value SRID:

**`0`**

That is a fixed “unspecified Cartesian reference” contract.

It is **not** `GEOMETRY(ANY)`.

### 2. Fixed SRID Actually Means Fixed

A Geometry value carrying SRID `4326` was forced into a `GEOMETRY(3857)` schema.

Spark rejected it:

**True**

The failure type was:

**`SparkRuntimeException`**

That is the point of embedding the SRID in the datatype: the schema can now reject a contradiction instead of silently admiring some bytes.

### 3. `ANY` Is Catalyst Admitting the Truth

Unioning Geometry values with SRIDs:

```text
[4326, 3857, 25830]
```

produced the common output type:

**`geometry(any)`**

The rows kept their original SRIDs.

```text
4326  → 4326
3857  → 3857
25830 → 25830
```

No reprojection happened. Spark changed the **common column type**, not the spatial coordinates.

Spark simply widened the type so the schema could describe the mixed result honestly.

### 4. Then the Multiverse Hit Disk

The storage boundary reduced the whole experiment to one very useful table:

```text
GEOMETRY(4326) → Parquet ✅
GEOMETRY(ANY)  → Parquet ❌
```

The fixed-SRID dataset successfully completed a Parquet round trip as:

**`geometry(4326)`**

with SRID:

**`4326`**

But writing `GEOMETRY(ANY)` to Parquet was rejected:

**True**

with:

**`AnalysisException`**

This is the architectural boundary the section describes.

Query-time type inference can say:

> “These are all geometries, but the rows disagree about the SRID.”

Persistent spatial storage eventually has to say:

> “This column has **this** spatial reference.”

> ## 🚀 Mission Verdict
> **Fixed SRIDs are contracts, `0` is an explicit unknown Cartesian reference, and `ANY` is a query-time escape hatch for honest mixed-SRID results.**
>
> `ANY` does not transform anything. It simply preserves disagreement.
>
> And when that disagreement reaches Parquet, storage refuses to pretend the column has one CRS.
>
> **The multiverse is perfectly legal in memory. Disk makes you pick a universe.**

### Evidence boundary

This notebook **runtime-proves Parquet** behavior. The same fixed-SRID persistence rule for **Delta** and **Iceberg** is stated in Spark 4.2 documentation, but those engines are not executed here.


## ✅ What This Run Actually Proved

```text
GEOMETRY(0) is fixed SRID 0, not ANY             ✅
fixed-SRID mismatch is rejected                  ✅
4326 + 3857 + 25830 union → GEOMETRY(ANY)        ✅
per-row SRIDs survive unchanged                  ✅
ANY performs no reprojection                     ✅
GEOMETRY(4326) → Parquet round trip              ✅
GEOMETRY(ANY)  → Parquet rejected                ✅
Delta fixed-SRID storage rule                    📚 documented
Iceberg fixed-SRID storage rule                  📚 documented
```

The important architectural result is not merely that `ANY` exists. It is that Spark can represent a mixed spatial reality during query analysis **without lying about it**, while persistent spatial storage eventually demands a single column-level CRS contract.

# 🛰️ Mission Handoff

We now understand the SRID contracts:

```text
4326
→ fixed reference

3857
→ another fixed reference

0
→ fixed but unspecified Cartesian reference

ANY
→ mixed valid references
```

But now a dangerous temptation appears:

```text
ST_SetSrid(...)
```

If a Geometry has the wrong SRID, surely we can just set the correct one and go home?

Surely.

Next mission:

> **`ST_SetSrid`: Change the Passport, Don't Move the Astronaut.** 🚨👨‍🚀

---

## 📚 Primary references

- Apache Spark 4.2 — Geospatial types and SRID/storage rules  
  https://spark.apache.org/docs/latest/sql-ref-geospatial-types.html

- Apache Spark 4.2 — SQL data types  
  https://spark.apache.org/docs/latest/sql-ref-datatypes.html

The notebook deliberately proves the Parquet boundary at runtime. Delta and Iceberg are described only as Spark-documented storage rules because their runtimes are not part of this stock-Spark lab.

In [8]:
spark.stop()
print("🌌 Spark stopped. The multiverse has been safely returned to RAM.")

🌌 Spark stopped. The multiverse has been safely returned to RAM.
